# 🧠 سيرفر الموديل على Colab (سحابي + مفتاح)

بنشغّل **الموديل على GPU بتاع كولاب**، وبيطلع **رابط عام + مفتاح**.
التطبيق عندك بيتصل بالموديل ده **بالمفتاح** — يعني كل عمليات الموديل **سحابية**.

## الخطوات
1. `Runtime` → `Change runtime type` → **T4 GPU** → `Save`
2. شغّل الخلايا بالترتيب (`Shift + Enter`)
3. في آخر خلية هيطلعلك **🔗 رابط السيرفر** و **🔑 المفتاح**
   → افتح التطبيق عندك → تاب **⚙️ الإعدادات** → نوع المحرك = `server` → الصق الرابط والمفتاح → **تطبيق وتشغيل**

> ⚠️ ده سيرفر مؤقت (كولاب المجاني مش دائم). كل ما تقفل الجلسة، تشغّله من جديد ويطلع **رابط ومفتاح جديدين**.

> 🔐 الرابط عام، بس محدش يقدر يولّد من غير **المفتاح**.

In [ ]:
# 1) نتأكد إن GPU شغّال
import torch
print("CUDA متاح؟", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠️ من غير GPU! ارجع: Runtime → Change runtime type → T4 GPU")
else:
    print("✅ GPU جاهز:", torch.cuda.get_device_name(0))

In [ ]:
# 2) تثبيت المكتبات (دقيقتين تقريبًا)
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q voicetut-tts soundfile pyloudnorm scipy lameenc fastapi uvicorn python-multipart requests hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # تنزيل أسرع بكتير للموديل (2.5GB)
print("✅ المكتبات اتثبّتت")

In [ ]:
# 3) هات الكود (أول مرة ينسخ، وبعدها بيعمل تحديث تلقائي)
%cd /content
import os

REPO = "https://github.com/pharmacygamila-ship-it/egyptian-voice-studio.git"
if os.path.isdir("project/.git"):
    %cd project
    !git pull -q
    print("✅ اتحدّث لآخر نسخة")
else:
    !git clone -q {REPO} project
    %cd project
    print("✅ اتنسخ لأول مرة")

In [ ]:
# 4) نزّل cloudflared (بيعمل نفق عام للسيرفر) — مرة واحدة
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# 4.5) (اختياري لكن موفّر للوقت) خزّن الموديل على Google Drive
#      أول جلسة هتنزّله من النت؛ الجلسات اللي بعدها هتحمّله من Drive (أسرع).
USE_DRIVE_CACHE = False   # ← خليها True لو عندك مساحة على Drive

import os
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/voice-studio/hf", exist_ok=True)
    os.environ["HF_HOME"] = "/content/drive/MyDrive/voice-studio/hf"
    print("✅ الموديل هيتخزّن على Drive/voice-studio/hf")
else:
    print("ℹ️ الموديل هيتنزّل كل جلسة. لو عايز توفّر وقت: خلي USE_DRIVE_CACHE = True")

In [ ]:
# 5) 🚀 شغّل سيرفر الموديل + النفق العام + ولّد مفتاح
import os, sys, re, time, secrets, subprocess, pathlib, requests

os.environ["TTS_ENGINE"] = "local"                    # الموديل يشتغل هنا على GPU
os.environ["TTS_DATA_DIR"] = "/content/project/data"
os.environ["TTS_OUTPUT_DIR"] = "/content/project/data/outputs"
TOKEN = "vs_" + secrets.token_urlsafe(24)              # 🔑 المفتاح
os.environ["TTS_SERVER_TOKEN"] = TOKEN

# 1) شغّل السيرفر في الخلفية
subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/project",
    stdout=open("/content/server.log", "a"), stderr=subprocess.STDOUT,
)
up = False
for _ in range(60):
    try:
        requests.get("http://127.0.0.1:8000/health", timeout=2); up = True; break
    except Exception:
        time.sleep(1)
print("✅ السيرفر شغّال" if up else "⚠️ السيرفر مش راضي يقوم — بص /content/server.log")
print("ℹ️ السيرفر بيرد فورًا دلوقتي — الموديل بيتحمّل في خلية التحميل المسبق الجاية.")

# 2) افتح نفق عام
pathlib.Path("/content/tunnel.log").write_text("")
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=open("/content/tunnel.log", "a"), stderr=subprocess.STDOUT,
)
URL = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                  pathlib.Path("/content/tunnel.log").read_text(errors="ignore"))
    if m:
        URL = m.group(0); break

print("\n" + "=" * 66)
print("🔗 رابط السيرفر :", URL or "⚠️ مش ظهر — بص /content/tunnel.log")
print("🔑 المفتاح       :", TOKEN)
print("=" * 66)
print("\nافتح التطبيق عندك → ⚙️ الإعدادات → نوع المحرك = server → الصق الاثنين → تطبيق وتشغيل")

In [ ]:
# 6) حمّل الموديل دلوقتي (تحميل مسبق) — التحميل ده بياخد شوية أول مرة (~2.5GB)
#    وقاعد كده بيخلي أول توليد من التطبيق يبقى سريع. استنى لحد ما يطبع "ok".
import requests
try:
    r = requests.post(f"{URL}/warmup", headers={"X-Auth-Token": TOKEN}, timeout=1800)
    print("الحالة:", r.status_code, "—", r.text[:300])
except Exception as e:
    print("⚠️ التحميل المسبق فشل:", e)

In [ ]:
# 7) اختبار سريع: توليد صوت من السيرفر وسماعه (بالمفتاح)
import requests
from IPython.display import Audio

r = requests.post(
    f"{URL}/tts",
    headers={"X-Auth-Token": TOKEN},
    json={"text": "أهلاً بيك، ده اختبار لسيرفر الموديل على كولاب", "speaker": "Sayed", "steps": 32},
    timeout=600,
)
if r.status_code == 200 and r.headers.get("content-type", "").startswith("audio"):
    open("/content/test.wav", "wb").write(r.content)
    print("✅ اتولّد — الحجم:", len(r.content), "بايت")
    Audio("/content/test.wav")
else:
    print("❌ خطأ:", r.status_code, r.text[:300])

## 📌 ملاحظات

- الرابط (`trycloudflare.com`) بيتغيّر كل مرة تشغّل الخلية — انسخه من جديد كل مرة.
- المفتاح **سرّي**: متشاركوش مع حد.
- لو النفق اتقفل، أعد تشغيل الخلية **5** (هيطلع رابط/مفتاح جديدين).
- عايز تشغّل التطبيق كله على كولاب (من غير مفتاح)؟ شوف نوتبوك `colab_free_gpu.ipynb`.
- للتشغيل التجاري المستمر 24 ساعة، كولاب المجاني مش مناسب — استخدم `deploy/modal_app.py` على Modal/RunPod.